# ASR Russian Numbers - Kaggle Inference Notebook

This notebook is intended for the competition submission workflow.

It:
- installs required dependencies,
- clones the public GitHub repository,
- downloads model assets from a GitHub Release,
- runs inference on the Kaggle test split,
- saves `submission.csv`.

In [ ]:
from pathlib import Path


GITHUB_REPO = "Kvazar-Off/ASR-TTS"
GIT_BRANCH = "group_project_1"


RELEASE_TAG = "v2.0.0"
CHECKPOINT_FILE = "averaged.pt"
CONFIG_FILE = "config.json"
NORM_STATS_FILE = "norm_stats.pt"

# Optional KenLM release asset. Set USE_KENLM = False to skip it.
USE_KENLM = True
KENLM_FILE = "numbers-3gram.binary"
LM_WEIGHT = 0.5
WORD_SCORE = 0.0
LM_TOP_PATHS = 8
BEAM_WIDTH = 5

# Auto-detect Kaggle vs local environment.
KAGGLE_INPUT = Path("/kaggle/input")
ON_KAGGLE = KAGGLE_INPUT.exists()

if ON_KAGGLE:
    # Kaggle execution: paths come from the competition's mounted dataset.
    AUDIO_DIR = "/kaggle/input/asr-2026-spoken-numbers-recognition-challenge"
    TEST_CSV  = f"{AUDIO_DIR}/test.csv"
    WORKDIR   = Path("/kaggle/working")
    SPLIT     = "test"
else:
    # Local fallback for debugging on a workstation.
    NB_DIR    = Path.cwd().resolve()
    AUDIO_DIR = str(NB_DIR / "data")
    TEST_CSV  = str(NB_DIR / "data" / "test.csv")
    WORKDIR   = NB_DIR / "_kaggle_run"
    SPLIT     = "test"  # change to "dev" if you only have dev data locally

REPO_DIR       = WORKDIR / "asr-russian-numbers"
CHECKPOINT_DIR = WORKDIR / "release_checkpoint"
OUTPUT_DIR     = WORKDIR / "submission_run"
OUTPUT_CSV     = OUTPUT_DIR / "submission.csv"

print("ON_KAGGLE:", ON_KAGGLE)
print("AUDIO_DIR:", AUDIO_DIR)
print("TEST_CSV :", TEST_CSV)
print("WORKDIR  :", WORKDIR)
print("SPLIT    :", SPLIT)

## 1. Install Dependencies

In [ ]:
import subprocess
import sys

def run(cmd: str) -> str:
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(f"Command failed: {cmd}")
    return result.stdout

# Audio I/O via soundfile avoids torchcodec/FFmpeg dependency.
run("pip install -q num2words soundfile tqdm")
if USE_KENLM:
    run("pip install -q https://github.com/kpu/kenlm/archive/master.zip")

print("Dependencies installed.")

## 2. Clone Source Code

In [ ]:
import os
import shutil
import stat

def _force_remove(func, path, *_):
    # Windows marks .git pack files as read-only; clear the bit and retry.
    try:
        os.chmod(path, stat.S_IWRITE)
        func(path)
    except Exception:
        pass

if REPO_DIR.exists():
    try:
        shutil.rmtree(REPO_DIR, onexc=_force_remove)
    except TypeError:
        # Python < 3.12 uses onerror instead of onexc.
        shutil.rmtree(REPO_DIR, onerror=_force_remove)

run(f"git clone --depth 1 --branch {GIT_BRANCH} https://github.com/{GITHUB_REPO}.git {REPO_DIR}")
print(f"Cloned into {REPO_DIR}")

## 3. Download Release Assets

In [ ]:
import os
import urllib.request

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

assets = [CHECKPOINT_FILE, CONFIG_FILE, NORM_STATS_FILE]
if USE_KENLM:
    assets.append(KENLM_FILE)

base_url = f"https://github.com/{GITHUB_REPO}/releases/download/{RELEASE_TAG}"

for asset_name in assets:
    url = f"{base_url}/{asset_name}"
    target_path = CHECKPOINT_DIR / asset_name
    print(f"Downloading {asset_name}...")
    urllib.request.urlretrieve(url, target_path)
    size_mb = os.path.getsize(target_path) / 1e6
    print(f"  saved to {target_path} ({size_mb:.2f} MB)")

## 4. Run Inference

In [ ]:
cmd = [
    sys.executable,
    "-u",
    str(REPO_DIR / "group_project_1" / "src" / "inference_kaggle.py"),
    "--split", SPLIT,
    "--data-root", AUDIO_DIR,
    "--checkpoint-dir", str(CHECKPOINT_DIR),
    "--checkpoint-name", CHECKPOINT_FILE,
    "--output-dir", str(OUTPUT_DIR),
    "--batch-size", "32",
    "--beam",
    "--beam-width", str(BEAM_WIDTH),
]

if USE_KENLM:
    cmd.extend([
        "--kenlm-model", str(CHECKPOINT_DIR / KENLM_FILE),
        "--lm-weight", str(LM_WEIGHT),
        "--word-score", str(WORD_SCORE),
        "--lm-top-paths", str(LM_TOP_PATHS),
    ])

print("Running:", " ".join(cmd))
print("CWD:", REPO_DIR)
print("-" * 60)

proc = subprocess.Popen(
    cmd,
    cwd=str(REPO_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,  # merge stderr into stdout so order is preserved
    text=True,
    bufsize=1,
)
captured = []
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="")
    captured.append(line)
returncode = proc.wait()

if returncode != 0:
    print("-" * 60)
    print("LAST 30 LINES OF OUTPUT (in case it scrolled away):")
    for line in captured[-30:]:
        print(line, end="")
    raise RuntimeError(f"Inference failed with exit code {returncode}.")

## 5. Inspect Submission

In [ ]:
import pandas as pd

submission = pd.read_csv(OUTPUT_CSV)
print(f"Saved {len(submission)} rows -> {OUTPUT_CSV}")
print("Unique predictions:", submission["transcription"].nunique())
submission.head(10)

## 6. Final Output Path

Use `/kaggle/working/submission_run/submission.csv` as the final competition submission file.